In [ ]:
import pandas as pd
import os
import openai
from typing import List, Dict, Optional
import json
from datetime import datetime
import textwrap
import requests
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("📚 Required libraries loaded successfully!")

In [ ]:
# 🔐 SECURE API CONFIGURATION
# Move API keys to environment variables for security

# Load API keys from environment variables
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
STRAVA_CLIENT_ID = os.getenv('STRAVA_CLIENT_ID')
STRAVA_CLIENT_SECRET = os.getenv('STRAVA_CLIENT_SECRET')
STRAVA_ACCESS_TOKEN = os.getenv('STRAVA_ACCESS_TOKEN')

# Validate required environment variables
if not OPENAI_API_KEY:
    print("⚠️  Warning: OPENAI_API_KEY not found in environment variables")
    print("   Please create a .env file with: OPENAI_API_KEY=your_key_here")

if not all([STRAVA_CLIENT_ID, STRAVA_CLIENT_SECRET]):
    print("⚠️  Warning: Strava credentials not found in environment variables")
    print("   Please add to .env file:")
    print("   STRAVA_CLIENT_ID=your_client_id")
    print("   STRAVA_CLIENT_SECRET=your_client_secret")
    print("   STRAVA_ACCESS_TOKEN=your_access_token")

# Set OpenAI API key
if OPENAI_API_KEY:
    openai.api_key = OPENAI_API_KEY
    print("✅ OpenAI API key loaded successfully")

CHAT_MODEL = "gpt-3.5-turbo"

In [ ]:
# 🤖 AI STORYTELLING FUNCTIONS

def query_openai(prompt, model=CHAT_MODEL):
    """Query OpenAI for AI-generated content."""
    if not OPENAI_API_KEY:
        return "Error: OpenAI API key not configured"
        
    try:
        response = openai.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a knowledgeable guide creating engaging stories about cycling routes. Focus on historical context, cultural significance, and interesting facts about places along cycling routes."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=500,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

def generate_route_story(locations, story_type="golden_age"):
    """Generate an engaging story about a cycling route through locations."""
    if not locations:
        return "No locations provided for story generation."
    
    if isinstance(locations, list):
        location_string = ", ".join(locations)
    else:
        location_string = locations
    
    if story_type == "golden_age":
        prompt = f"""Create a vivid narrative of a cyclist's journey through these locations during the Dutch Golden Age: {location_string}
        
        Requirements:
        - Start and end at the first location
        - Weave in historical events, notable figures, and cultural significance
        - Write as if the reader is experiencing the journey firsthand
        - Include sensory details and atmospheric descriptions
        - Keep it engaging and historically informed"""
        
    elif story_type == "modern":
        prompt = f"""Tell the story of a modern cyclist's journey through: {location_string}
        
        Include:
        - Contemporary landmarks and features
        - Local culture and modern significance
        - Cycling-specific details (routes, terrain, notable cycling spots)
        - Personal reflection on the journey"""
    
    return query_openai(prompt)

print("🤖 AI storytelling functions loaded")

In [ ]:
# 🚴‍♂️ STRAVA API SETUP
# Install stravalib if not already installed
try:
    from stravalib import Client
    print("✅ stravalib already installed")
except ImportError:
    print("📦 Installing stravalib...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "stravalib"])
    from stravalib import Client
    print("✅ stravalib installed successfully")

# Strava API Configuration
class StravaAPI:
    def __init__(self):
        self.client = None
        self.access_token = STRAVA_ACCESS_TOKEN
        
    def authenticate(self):
        """Initialize Strava client with access token."""
        if not self.access_token:
            print("❌ No Strava access token found")
            print("   Visit: https://www.strava.com/settings/api")
            print("   Create an app and get your access token")
            return False
            
        try:
            self.client = Client(access_token=self.access_token)
            athlete = self.client.get_athlete()
            print(f"✅ Successfully authenticated as: {athlete.firstname} {athlete.lastname}")
            return True
        except Exception as e:
            print(f"❌ Strava authentication failed: {e}")
            return False
    
    def get_recent_activities(self, limit=10):
        """Fetch recent activities from Strava."""
        if not self.client:
            print("❌ Not authenticated with Strava")
            return []
            
        try:
            activities = list(self.client.get_activities(limit=limit))
            print(f"✅ Found {len(activities)} recent activities")
            return activities
        except Exception as e:
            print(f"❌ Error fetching activities: {e}")
            return []
    
    def get_activity_streams(self, activity_id, stream_types=['latlng']):
        """Get activity streams (GPS coordinates) for route mapping."""
        if not self.client:
            return None
            
        try:
            streams = self.client.get_activity_streams(activity_id, types=stream_types)
            return streams
        except Exception as e:
            print(f"❌ Error fetching streams for activity {activity_id}: {e}")
            return None

# Initialize Strava API
strava_api = StravaAPI()
print("🔧 Strava API client initialized")

In [ ]:
# 🌍 ROUTE PROCESSING & LOCATION EXTRACTION

def extract_locations_from_coordinates(lat_lng_points, max_locations=10):
    """
    Extract location names from GPS coordinates using reverse geocoding.
    This is a simplified version - in production, you'd use proper geocoding APIs.
    """
    if not lat_lng_points:
        return []
    
    # For demo purposes, we'll use a subset of points
    sample_points = lat_lng_points[::len(lat_lng_points)//max_locations] if len(lat_lng_points) > max_locations else lat_lng_points
    
    locations = []
    try:
        # This would typically use a geocoding service like Google Maps, OpenStreetMap, etc.
        # For now, we'll return some sample Dutch locations as a placeholder
        locations = ["The Hague", "Delft", "Rotterdam", "Gouda", "Utrecht"][:len(sample_points)]
        print(f"📍 Extracted {len(locations)} locations from route coordinates")
    except Exception as e:
        print(f"❌ Error extracting locations: {e}")
        
    return locations

class RouteProcessor:
    """Process Strava routes and extract meaningful location data."""
    
    def __init__(self, strava_api):
        self.strava_api = strava_api
        
    def get_route_from_activity(self, activity_id):
        """Extract route data from a specific Strava activity."""
        print(f"🔍 Processing activity: {activity_id}")
        
        streams = self.strava_api.get_activity_streams(activity_id, ['latlng'])
        if not streams or 'latlng' not in streams:
            print("❌ No GPS data found for this activity")
            return None
            
        coordinates = streams['latlng'].data
        locations = extract_locations_from_coordinates(coordinates)
        
        return {
            'activity_id': activity_id,
            'coordinates': coordinates,
            'locations': locations,
            'total_points': len(coordinates)
        }
    
    def get_route_from_recent_activity(self, activity_type='Ride'):
        """Get route data from the most recent cycling activity."""
        if not self.strava_api.client:
            print("❌ Strava not authenticated")
            return None
            
        activities = self.strava_api.get_recent_activities(limit=20)
        cycling_activities = [a for a in activities if str(a.type).lower() in ['ride', 'cycling']]
        
        if not cycling_activities:
            print("❌ No recent cycling activities found")
            return None
            
        latest_ride = cycling_activities[0]
        print(f"🚴‍♂️ Using latest ride: {latest_ride.name} ({latest_ride.start_date_local})")
        
        return self.get_route_from_activity(latest_ride.id)

# Initialize route processor
route_processor = RouteProcessor(strava_api)
print("🗺️ Route processor initialized")

In [ ]:
# 🎯 MAIN EXECUTION: STRAVA TELL ME MORE

def run_strava_tell_me_more(use_real_strava_data=True, fallback_to_mock=True):
    """
    Main function to run the Strava Tell Me More application.
    
    Args:
        use_real_strava_data: Try to use real Strava data first
        fallback_to_mock: Fall back to mock data if Strava fails
    """
    print("🚴‍♂️ STRAVA TELL ME MORE - Starting...")
    print("=" * 60)
    
    route_data = None
    locations = []
    data_source = "unknown"
    
    # Try to use real Strava data first
    if use_real_strava_data:
        print("🔐 Attempting Strava authentication...")
        if strava_api.authenticate():
            print("📱 Fetching route data from Strava...")
            route_data = route_processor.get_route_from_recent_activity()
            
            if route_data:
                locations = route_data['locations']
                data_source = "strava"
                print(f"✅ Using real Strava data: {len(locations)} locations")
            else:
                print("⚠️  No route data found from Strava")
        else:
            print("⚠️  Strava authentication failed")
    
    # Fallback to mock data if needed
    if not locations and fallback_to_mock:
        print("🎭 Falling back to mock Dutch cycling route data...")
        locations = ["Segbroek", "Kraayenstein", "Kwintsheul", "Schipluiden", 
                    "Berkel en Rodenrijs", "Pijnacker", "Leidschendam", "Marlot", "Haagse Bos"]
        data_source = "mock"
        print(f"✅ Using mock data: {len(locations)} locations")
    
    if not locations:
        print("❌ No location data available - cannot generate stories")
        return None
    
    # Create route summary
    route_summary = {
        'data_source': data_source,
        'locations': locations,
        'location_count': len(locations),
        'generated_at': datetime.now().isoformat()
    }
    
    if route_data:
        route_summary.update({
            'activity_id': route_data['activity_id'],
            'total_gps_points': route_data['total_points']
        })
    
    print(f"\n🗺️  ROUTE SUMMARY")
    print(f"   Data Source: {data_source.upper()}")
    print(f"   Locations: {len(locations)}")
    print(f"   Route: {' → '.join(locations[:3])}{'...' if len(locations) > 3 else ''}")
    
    return route_summary

# Run the main application
route_info = run_strava_tell_me_more()

In [ ]:
# 📚 GENERATE CYCLING STORIES

def display_route_story(route_info, story_type="golden_age"):
    """Generate and display an engaging story about the cycling route."""
    if not route_info or not route_info.get('locations'):
        print("❌ No route information available for story generation")
        return
    
    locations = route_info['locations']
    data_source = route_info['data_source']
    
    print(f"\n📖 GENERATING {story_type.upper().replace('_', ' ')} STORY")
    print(f"🗺️  Route: {' → '.join(locations)}")
    print(f"📍 Data Source: {data_source.upper()}")
    print("=" * 80)
    
    # Generate the story
    story = generate_route_story(locations, story_type)
    
    # Display with nice formatting
    print(textwrap.fill(story, width=100))
    print("=" * 80)
    
    # Save story to file
    story_data = {
        'story': story,
        'story_type': story_type,
        'route_info': route_info,
        'generated_at': datetime.now().isoformat()
    }
    
    filename = f"cycling_story_{story_type}_{data_source}.json"
    try:
        with open(filename, 'w') as f:
            json.dump(story_data, f, indent=2)
        print(f"💾 Story saved to: {filename}")
    except Exception as e:
        print(f"❌ Error saving story: {e}")
    
    return story

# Generate different types of stories if we have route info
if route_info:
    print("\n🎭 STORY GENERATION")
    print("=" * 60)
    
    # Generate Golden Age story
    golden_age_story = display_route_story(route_info, "golden_age")
    
    print("\n" + "="*60)
    
    # Generate Modern story
    modern_story = display_route_story(route_info, "modern")
else:
    print("❌ No route information available - skipping story generation")

In [ ]:
# define function to query openai
def query_2(route_locations_list, model=chat_model):
    try:
        response = openai.chat.completions.create(
            model=model,  # Now you can specify different models
            messages=[
                {"role": "user", "content": route_locations_list}
            ],
            max_tokens=100,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"


In [ ]:
# 🔧 SETUP INSTRUCTIONS & ENVIRONMENT FILE CREATION

def create_env_template():
    """Create a template .env file with instructions."""
    env_template = """# Strava Tell Me More - Environment Variables
# Copy this file to .env and fill in your actual API keys

# OpenAI API Key (required for story generation)
# Get from: https://platform.openai.com/api-keys
OPENAI_API_KEY=your_openai_api_key_here

# Strava API Credentials (required for real route data)
# Get from: https://www.strava.com/settings/api
STRAVA_CLIENT_ID=your_strava_client_id
STRAVA_CLIENT_SECRET=your_strava_client_secret
STRAVA_ACCESS_TOKEN=your_strava_access_token

# Instructions:
# 1. Create a Strava API application at https://www.strava.com/settings/api
# 2. Get your Client ID and Client Secret
# 3. For Access Token, you'll need to complete OAuth flow or use temporary token
# 4. Replace the placeholder values above with your actual credentials
# 5. Save this file as .env (without .template extension)
"""
    
    try:
        with open('.env.template', 'w') as f:
            f.write(env_template)
        print("✅ Created .env.template file")
        print("📝 Instructions:")
        print("   1. Copy .env.template to .env")
        print("   2. Fill in your actual API keys")
        print("   3. Keep .env file secure and don't commit it to git")
    except Exception as e:
        print(f"❌ Error creating template: {e}")

def display_setup_instructions():
    """Display setup instructions for users."""
    print("🛠️  SETUP INSTRUCTIONS")
    print("=" * 60)
    print("To use real Strava data, you need to:")
    print()
    print("1. 🔑 Get Strava API Credentials:")
    print("   • Visit: https://www.strava.com/settings/api")
    print("   • Create a new API application")
    print("   • Note your Client ID and Client Secret")
    print()
    print("2. 🎫 Get OpenAI API Key:")
    print("   • Visit: https://platform.openai.com/api-keys")
    print("   • Create a new API key")
    print()
    print("3. 📝 Create .env file:")
    print("   • Copy .env.template to .env")
    print("   • Fill in your actual API keys")
    print()
    print("4. 🚀 Install dependencies:")
    print("   • pip install stravalib python-dotenv openai")
    print()
    print("Current Status:")
    if OPENAI_API_KEY:
        print("   ✅ OpenAI API configured")
    else:
        print("   ❌ OpenAI API not configured")
    
    if STRAVA_ACCESS_TOKEN:
        print("   ✅ Strava API configured")
    else:
        print("   ❌ Strava API not configured")

# Create template and show instructions
create_env_template()
display_setup_instructions()

In [16]:
# Get unique values from country and towns
unique_locations = route_locations.drop_duplicates(subset=['country', 'towns'])
unique_locations_list = unique_locations.values.tolist()

print(f"Original locations: {len(route_locations)}")
print(f"Unique locations: {len(unique_locations)}")
print("\nUnique locations:")
for country, town in unique_locations_list:
    print(f"  • {town}, {country}")

Original locations: 9
Unique locations: 9

Unique locations:
  • Segbroek, The Netherlands
  • Kraayenstein, The Netherlands
  • Kwintsheul, The Netherlands
  • Schipluiden, The Netherlands
  • Berkel en Rodenrijs, The Netherlands
  • Pijnacker, The Netherlands
  • Leidschendam, The Netherlands
  • Marlot, The Netherlands
  • Haagse Bos, The Netherlands


In [17]:
# Function to query OpenAI for unique route locations
def query_unique_route_locations(unique_locations_df, model=chat_model):
    """
    Query OpenAI for information about each unique location in the route.
    
    Args:
        unique_locations_df: DataFrame with unique 'country' and 'towns' columns
        model: OpenAI model to use
    
    Returns:
        DataFrame with original data plus AI-generated descriptions
    """
    results = []
    
    print(f"🌍 Querying AI for {len(unique_locations_df)} unique locations...")
    print("=" * 50)
    
    for index, row in unique_locations_df.iterrows():
        town = row['towns']
        country = row['country']
        
        # Create a prompt for each unique location
        prompt = f"Tell me about {town}, {country}. Include interesting facts about its history, culture, or notable features. Keep it brief and engaging."
        
        print(f"Processing {index + 1}/{len(unique_locations_df)}: {town}")
        
        # Query OpenAI
        response = query_openai(prompt, model)
        
        # Store results
        result = {
            'country': country,
            'towns': town,
            'ai_description': response,
            'generated_at': datetime.now().isoformat()
        }
        results.append(result)
        
        # Small delay to avoid rate limiting
        import time
        time.sleep(0.5)
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    print(f"\n✅ Generated descriptions for {len(results_df)} unique locations!")
    return results_df

# Test the function with unique locations
unique_route_descriptions = query_unique_route_locations(unique_locations)

🌍 Querying AI for 9 unique locations...
Processing 1/9: Segbroek
Processing 2/9: Kraayenstein
Processing 3/9: Kwintsheul
Processing 4/9: Schipluiden
Processing 5/9: Berkel en Rodenrijs
Processing 6/9: Pijnacker
Processing 7/9: Leidschendam
Processing 8/9: Marlot
Processing 9/9: Haagse Bos

✅ Generated descriptions for 9 unique locations!


In [18]:
# Display the unique route descriptions
def display_unique_route_descriptions(df):
    """
    Display the AI-generated descriptions for unique locations.
    """
    print("🏙️  UNIQUE ROUTE LOCATION DESCRIPTIONS")
    print("=" * 60)
    
    for index, row in df.iterrows():
        town = row['towns']
        country = row['country']
        description = row['ai_description']
        generated_at = row['generated_at']
        
        print(f"\n🏙️  {town.upper()}, {country}")
        print("-" * (len(town) + len(country) + 6))
        print(description)
        print(f"\nGenerated: {generated_at}")
        print("=" * 60)

# Display the results
display_unique_route_descriptions(unique_route_descriptions)

🏙️  UNIQUE ROUTE LOCATION DESCRIPTIONS

🏙️  SEGBROEK, The Netherlands
-----------------------------
Segbroek is a district located in The Hague, Netherlands, known for its diverse community and lively atmosphere. The area is home to a mix of modern architecture and historic buildings, creating a unique blend of old and new. 

One notable feature of Segbroek is its beautiful parks and green spaces, such as the Westduinpark and Bosjes van Poot, which offer residents and visitors a peaceful retreat from city life. The district also boasts a variety of cultural attractions, including museums,

Generated: 2025-08-02T11:40:11.357505

🏙️  KRAAYENSTEIN, The Netherlands
---------------------------------
Kraayenstein is a neighborhood located in The Hague, Netherlands. Originally a rural area, it has developed into a residential area with a mix of modern architecture and green spaces. The neighborhood is known for its proximity to the Westduinpark, a large nature reserve perfect for hiking and c

In [19]:
# Save the unique route descriptions
def save_unique_route_descriptions(df, filename="unique_route_descriptions.json"):
    """
    Save the unique route descriptions to a JSON file.
    """
    try:
        # Convert DataFrame to list of dictionaries
        data = df.to_dict('records')
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"✅ Unique route descriptions saved to {filename}")
        
    except Exception as e:
        print(f"❌ Error saving file: {e}")

# Save the results
save_unique_route_descriptions(unique_route_descriptions)

✅ Unique route descriptions saved to unique_route_descriptions.json


In [20]:
# Complete workflow with unique values
print("🚴‍♂️ Strava Route - Unique Location AI Descriptions")
print("=" * 60)

# Get unique locations
unique_locations = route_locations.drop_duplicates(subset=['country', 'towns'])
print(f"Found {len(unique_locations)} unique locations from {len(route_locations)} total locations")

# Generate descriptions for unique locations
unique_route_descriptions = query_unique_route_locations(unique_locations)

# Display results
display_unique_route_descriptions(unique_route_descriptions)

# Save to file
save_unique_route_descriptions(unique_route_descriptions)

print("\n🎉 All unique route locations processed!")
print(f"📊 Summary: {len(unique_locations)} unique locations processed")
print(f"💾 Results saved to: unique_route_descriptions.json")

🚴‍♂️ Strava Route - Unique Location AI Descriptions
Found 9 unique locations from 9 total locations
🌍 Querying AI for 9 unique locations...
Processing 1/9: Segbroek
Processing 2/9: Kraayenstein
Processing 3/9: Kwintsheul
Processing 4/9: Schipluiden
Processing 5/9: Berkel en Rodenrijs
Processing 6/9: Pijnacker
Processing 7/9: Leidschendam
Processing 8/9: Marlot
Processing 9/9: Haagse Bos

✅ Generated descriptions for 9 unique locations!
🏙️  UNIQUE ROUTE LOCATION DESCRIPTIONS

🏙️  SEGBROEK, The Netherlands
-----------------------------
Segbroek is a district located in The Hague, Netherlands. It is known for its diverse population and vibrant cultural scene. The district has a rich history, with many historic buildings and landmarks to explore. One notable feature of Segbroek is its beautiful parks and green spaces, providing residents and visitors with plenty of opportunities for outdoor activities. The district also has a strong sense of community, with various events and festivals hel